# Models

This notebook contains work done on the models, including code and logs of running each component model and metrics.

Functional pieces of code remain in their respective Python modules and are imported here when necessary.

## Running the code

As per the Dataset Audit notebook, the following shell commands need to be run in the `models` directory to be able to run the code:

```bash
# Set up virtual environment
python -m venv venv

# Install required packages
pip install -r requirements.txt

# Set up kernel
python -m ipykernel install --user --name=<Jupyter_Kernel_Name>
```

This should be followed by selecting the kernel in the Jupyter notebook.

## Dataset loading and preparation

To be able to load and prepare the dataset, we need to import modules from the `dataset` directory,
which is in the current folder's parent directory.

To address this, we can add the parent directory to the current PATH:

In [1]:
import os
import sys

parent = os.path.dirname(os.getcwd())
sys.path.insert(0, parent)

With this done, we can load the dataset, similarly to what we did in the Dataset Audit, 
and then feed it into the preprocessor and splitter pipeline to get the expected format.

In [3]:
import dataset.dataset_loader
from dataset.preprocessor import DatasetPipeline

corpus = dataset.dataset_loader.load_brown_corpus_df()
dataset = DatasetPipeline(corpus).get()

dataset

,raw_text,label,subset,fold_id,tokenized_text,tokenized_tags
0,She/pps could/md have/hv found/vbn out/rp my/p...,romance,test,-1,"[She, could, have, found, out, my, first, name...","[pps, md, hv, vbn, rp, pp$, od, nn, ,, in, nn,..."
1,I/ppss wish/vb he/pps was/bedz with/in me/ppo ...,romance,test,-1,"[I, wish, he, was, with, me, now, ,, she, thou...","[ppss, vb, pps, bedz, in, ppo, rb, ,, pps, vbd..."
2,"And/cc ,/, still/rb ,/, they/ppss did/dod not/...",romance,test,-1,"[And, ,, still, ,, they, did, not, have, to, b...","[cc, ,, rb, ,, ppss, dod, *, hv, to, be, ql, j..."
3,He/pps was/bedz mad/jj about/in stars/nns at/i...,romance,test,-1,"[He, was, mad, about, stars, at, the, age, of,...","[pps, bedz, jj, in, nns, in, at, nn, in, cd, .]"
4,She/pps went/vbd to/in the/at father/nn and/cc...,romance,test,-1,"[She, went, to, the, father, and, found, he, h...","[pps, vbd, in, at, nn, cc, vbd, pps, hvd, vbn,..."
...,...,...,...,...,...,...
56665,Alec/np was/bedz growing/vbg more/ql and/cc mo...,mystery,train,4,"[Alec, was, growing, more, and, more, skeptica...","[np, bedz, vbg, ql, cc, ql, jj, .]"
56666,--/-- to/to lunge/vb anxiously/rb through/in c...,mystery,train,4,"[--, to, lunge, anxiously, through, crowded, s...","[--, to, vb, rb, in, vbn, nns, in, nn, nns, cc..."
56667,She/pps looked/vbd at/in me/ppo provocatively/...,mystery,train,4,"[She, looked, at, me, provocatively, .]","[pps, vbd, in, ppo, rb, .]"
56668,"``/`` Oh/uh ,/, come/vb on/rp ,/, snap/vb out/...",mystery,train,4,"[``, Oh, ,, come, on, ,, snap, out, of, it, !, !]","[``, uh, ,, vb, rp, ,, vb, in, in, ppo, ., .]"


In the dataframe returned above, we can see what the raw text is successfully tokenised (as per the dataset audit conclusion),
and each row is assigned to either the test set, or one of 5 folds in our $K$-fold cross-validation schema training set.

With the dataset in this shape, we can start looking at the models.

## Baseline Model

For the baseline model, as proposed previously, this project uses a majority-class tagger,
i.e. each word gets assigned the tag that it most frequently appears as in the training data.

We can use the class `MajorityTagger` I created in the `majority_tagger.py` module next to this notebook.

The `.fit` method of the `MajorityTagger` class takes in a dataset, and allows
excluding a specific fold for the cross-validation design.

We can go through the dataset excluding each of the 5 folds and gather the accuracy and macro-F1 scores for each.

In [4]:
from majority_tagger import MajorityTagger
from metrics import Metrics

baseline = MajorityTagger()
metrics = Metrics(baseline, dataset)

accuracy = []
macro_f1 = []

for fold in range(5):
    baseline.fit(dataset, fold)
    acc, _, _ = metrics.get_val_accuracy(fold, quiet=True)
    accuracy.append(acc)
    mf1, _ = metrics.get_val_macro_f1(fold, quiet=True)
    macro_f1.append(mf1)

print("Accuracy scores:", accuracy)
print("Macro-F1 scores:", macro_f1)

Accuracy scores: [0.9123290781982449, 0.9126533005423542, 0.9120445430497482, 0.9115754409872057, 0.9126752809352634]
Macro-F1 scores: [0.8466126933061805, 0.8373434796825943, 0.8308498934634351, 0.8335407468535522, 0.8433779049810051]


With these scores, we can calculate the mean and standard deviation across all folds using Python's built-in statistics module:

In [5]:
import statistics

print(f"Mean accuracy: {statistics.mean(accuracy)}, with standard deviation: {statistics.stdev(accuracy)}")
print(f"Mean Macro-F1: {statistics.mean(macro_f1)}, with standard deviation: {statistics.stdev(macro_f1)}")

Mean accuracy: 0.9122555287425633, with standard deviation: 0.00046012447989008987
Mean Macro-F1: 0.8383449436573535, with standard deviation: 0.006594418063271458


From these results, we can observe that while the accuracy is reasonably high (around 91%) with a very low standard deviation,
the macro-F1 score is noticably lower (at only about 84%) with a standard deviation about an order of magnitude higher.

This supports the dataset audit's hypothesis that due to the long-tailed, outlier-heavy distribution,
accuracy won't be a metric sufficient by itself.

Next, we can fit the baseline model to the entirety of the train set and see the overall consolidated model metrics
on the separate test set.

In [6]:
baseline.fit(dataset)

accuracy, _, _ = metrics.get_test_accuracy()
macro_f1, scores = metrics.get_test_macro_f1()

Metrics: accuracy 0.9155 (212679/232316 tagged correct) on test set
Metrics: Macro-F1 0.8301


The test set scores of about 92% accuracy and about 83% macro-F1 score are in line with what is expected
based on the $K$-fold cross-validation fitting shown previously.

To see where the macro-F1 score struggles, we can analyse the individual per-tag scores
(individual F1 score for each target variable class).

In [10]:
print(scores)

{'jjt': 0.8980716253443526, 'bez': 1.0, 'rn': 0.0, 'ber*': 0.6666666666666666, 'cd': 0.948112394414756, '(': 0.9946638207043756, 'abx': 1.0, 'bed*': 1.0, 'wrb': 0.9622030237580994, 'hvd': 0.9741337237676916, 'pp$': 0.964633068081344, 'ex': 0.8875968992248062, 'be': 1.0, 'nps$': 0.4444444444444444, 'do': 1.0, 'ppls': 1.0, 'doz': 1.0, '--': 1.0, ',': 0.9998715588474547, 'at': 0.9976352201257862, 'abl': 0.0, 'dod': 1.0, 'dod*': 1.0, 'vbd': 0.7432950191570882, 'dts': 0.9948822927328557, 'dt$': 1.0, 'rb': 0.8610388697546918, ':': 0.9739413680781759, 'nn': 0.8913291579679888, '*': 0.9994568169473113, 'np$': 0.799140708915145, 'ber': 0.9994538503549972, 'np': 0.9126806431915326, 'nr$': 0.96, '.': 0.9995494203907754, 'cs': 0.8287018658197698, 'pp$$': 0.75, 'bez*': 0.9056603773584906, 'pn$': 0.9166666666666666, 'md': 0.991496934941665, 'do*': 1.0, 'bedz': 1.0, 'wql': 0.0, 'ppo': 0.7140458869613878, 'hvg': 1.0, 'wdt': 0.9977944419938244, 'vbg': 0.8944807451689142, 'bed': 1.0, 'hv*': 1.0, 'pn': 0

While some of the less ambiguous and more frequent tags achieve a perfect F1 score of $1.0$,
some of the outlier tags remain at $0.0$ which will inevitably drag the arithmetic mean value down.

Some of the tags common with less clear-cut words also have noticably lower scores, for example `vbd`, a past-tense verb,
only achieves an F1 score of about $0.743$ (as words that are past-tense verbs in some contexts can be, for example, nouns in others).

The notably lower macro-F1 score as well as analysing the individual per-tag F1 scores indicates that
the baseline majority-class tagger is capable of dealing with some of the more common tags (which dominate this dataset) 
and overall tags the majority of them correctly, but struggles with outliers and doesn't achieve a very high score
in frequent but commonly somewhat ambiguous tags.